In [4]:
import pandas as pd
import numpy as np
import os
# --- Load data
directory = r"C:\Users\20234503\Desktop\Research\Weather and Strikes\\"
df = pd.read_csv(directory + r"Project\Data\Weather_Labour_Unrest_Event.csv")

# --- Filter to summer months
df["date"] = pd.to_datetime(df["date"])
summer = df[df["date"].dt.month.isin([6, 7, 8])].copy()

# --- Variables
temp_col = "nn_tavg"
extreme_cols = ["tavg_norm_above_3SD"]

# Create indicator for extreme days (> 2.5 SD)
summer["extreme"] = summer[extreme_cols].sum(axis=1).clip(0, 1)

# --- Define groups
groups = {
    "All Summer Days": summer,
    "\hspace{0.25cm} Cooler Counties": summer[summer["CDD_median"] == 1],
    "\hspace{0.25cm} Warmer Counties": summer[summer["CDD_median"] == 2],
}

# --- Compute summary stats for *extreme days only*
summary = []
for name, subset in groups.items():
    extreme_subset = subset[subset["extreme"] == 1]
    stats = {
        "Group": name,
        "Mean": extreme_subset[temp_col].mean(),
        "SD": extreme_subset[temp_col].std(),
        "Min": extreme_subset[temp_col].min(),
        "N Obs": len(extreme_subset),
        "% of Total Days": 100 * len(extreme_subset) / len(subset)
    }
    summary.append(stats)

summary_df = pd.DataFrame(summary).round(2)

print(summary_df)

# --- Directory to save
save_dir = r"C:\Users\20234503\Desktop\Research\Weather and Strikes\Project\Results_2"
os.makedirs(save_dir, exist_ok=True)
out_path = os.path.join(save_dir, "summary_extreme_summer.tex")

# --- Column order + labels
cols_display_order = ["Group", "Mean", "SD", "Min", "N Obs", "% of Total Days"]

nice_labels = {
    "Group": "Group",
    "Mean": "Mean Temp ($^{\\circ}$C)",
    "SD": "SD",
    "Min": "Min Temp ($^{\\circ}$C)",
    "N Obs": "N Obs",
    "% of Total Days": "\\% of Total Days"
}

# LaTeX tabular column specification: first column left-aligned, rest centred
header_cols = "l" + "c" * (len(cols_display_order) - 1)

lines = []
lines += [
    r"\begin{table}[htbp]",
    r"\centering",
    r"\caption{Summary of Extreme Summer Heat Shocks}",
    r"\label{tab:summary_extreme_summer}",
    r"\begin{tabular}{" + header_cols + r"}",
    r"\toprule",
    r" & " + " & ".join(nice_labels[col] for col in cols_display_order) + r" \\",
    r"\midrule"
]

# --- Table body
for _, row in summary_df.iterrows():
    row_vals = []

    # First column: group name as text
    row_vals.append(str(row["Group"]))

    # Remaining columns: numeric formatting
    for col in cols_display_order[1:]:
        val = row[col]

        if isinstance(val, (int, float, np.floating)):
            if col == "% of Total Days":
                # add percent sign, escaped for LaTeX
                cell = f"{val:.2f}\\%"
            else:
                cell = f"{val:.2f}"
        else:
            cell = str(val)

        row_vals.append(cell)

    lines.append(" & ".join(row_vals) + r" \\")

lines += [
    r"\bottomrule",
    r"\end{tabular}",
    r"\begin{minipage}{0.85\linewidth}",
    r"\tiny",
    r"\setstretch{0.9}",
    r"\textit{Notes:} This table presents summary statistics for extreme summer heat shocks",
    r" (June--August) based on county-day data. Extreme days are defined as those where the",
    r" average temperature is more than 3 standard deviations above the long-run norm.",
    r" Panel reports the mean, standard deviation, minimum, number of observations, and the",
    r" share of days classified as extreme within each group.",
    r"\end{minipage}",
    r"\end{table}"
]

with open(out_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print(f"LaTeX table saved to:\n{out_path}")

<>:22: SyntaxWarning: invalid escape sequence '\h'
<>:23: SyntaxWarning: invalid escape sequence '\h'
<>:22: SyntaxWarning: invalid escape sequence '\h'
<>:23: SyntaxWarning: invalid escape sequence '\h'
C:\Users\20234503\AppData\Local\Temp\ipykernel_7348\4215492916.py:22: SyntaxWarning: invalid escape sequence '\h'
  "\hspace{0.25cm} Cooler Counties": summer[summer["CDD_median"] == 1],
C:\Users\20234503\AppData\Local\Temp\ipykernel_7348\4215492916.py:23: SyntaxWarning: invalid escape sequence '\h'
  "\hspace{0.25cm} Warmer Counties": summer[summer["CDD_median"] == 2],


                             Group   Mean    SD    Min  N Obs  % of Total Days
0                  All Summer Days  29.50  2.96  21.39    410             0.18
1  \hspace{0.25cm} Cooler Counties  27.90  2.66  21.39    251             0.19
2  \hspace{0.25cm} Warmer Counties  32.02  1.03  30.61    159             0.18
LaTeX table saved to:
C:\Users\20234503\Desktop\Research\Weather and Strikes\Project\Results_2\summary_extreme_summer.tex


In [ ]:
# Appendix Heat Shocks by Total & All Seasons #
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Load data
directory = r"C:\Users\20234503\Desktop\Research\Weather and Strikes"
df = pd.read_csv(directory + r"\Data\Weather Data\Weather_Labour_Unrest_Event.csv")

# ---------- 0) Clean columns ----------
df.columns = (df.columns
                .str.strip()
                .str.lower()
                .str.replace(r"\s+", "_", regex=True))

# ---------- 1) Season variable ----------
if 'month' not in df.columns:
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        df['month'] = df['date'].dt.month
    else:
        raise KeyError("Need 'date' or 'month' to compute seasons.")

df['season'] = np.select(
    [
        df['month'].isin([12, 1, 2]),   # Winter
        df['month'].isin([3, 4, 5]),    # Spring
        df['month'].isin([6, 7, 8]),    # Summer
        df['month'].isin([9, 10, 11]),  # Fall
    ],
    [1, 2, 3, 4],
    default=np.nan
)

# ---------- 2) Variables & extreme indicator ----------
temp_col = "nn_tavg"
extreme_cols = ["tavg_norm_above_3sd"]

# Indicator for extreme days (>= 3 SD) for ALL days
df["extreme"] = df[extreme_cols].sum(axis=1).clip(0, 1)

# Seasonal subsets (carry the 'extreme' column automatically)
winter = df[df["season"] == 1].copy()
spring = df[df["season"] == 2].copy()
summer = df[df["season"] == 3].copy()
fall   = df[df["season"] == 4].copy()

# ---------- 3) Define groups ----------
groups = {
    "All Days": df,
    "All Winter Days": winter,
    "All Spring Days": spring,
    "All Summer Days": summer,
    "All Fall Days": fall,
}

# ---------- 4) Compute summary stats for *extreme days only* ----------
summary = []
for name, subset in groups.items():
    # filter to extreme days within this group
    extreme_subset = subset[subset["extreme"] == 1]

    stats = {
        "Group": name,
        "Mean": extreme_subset[temp_col].mean(),
        "SD":   extreme_subset[temp_col].std(),
        "Min":  extreme_subset[temp_col].min(),
        "N Obs": len(extreme_subset),
        "% of Total Days": 100 * len(extreme_subset) / len(subset)
    }
    summary.append(stats)

summary_df = pd.DataFrame(summary).round(2)
print(summary_df)

# --- Directory to save
save_dir = r"C:\Users\20234503\Desktop\Research\Weather and Strikes\Project\Results_2\Appendix"
os.makedirs(save_dir, exist_ok=True)
out_path = os.path.join(save_dir, "summary_extreme_all.tex")

# --- Column order + labels
cols_display_order = ["Group", "Mean", "SD", "Min", "N Obs", "% of Total Days"]

nice_labels = {
    "Group": "Group",
    "Mean": "Mean Temp ($^{\\circ}$C)",
    "SD": "SD",
    "Min": "Min Temp ($^{\\circ}$C)",
    "N Obs": "N Obs",
    "% of Total Days": "\\% of Total Days"
}

# LaTeX tabular column specification: first column left-aligned, rest centred
header_cols = "l" + "c" * (len(cols_display_order) - 1)

lines = []
lines += [
    r"\begin{table}[htbp]",
    r"\centering",
    r"\caption{Summary of Extreme Summer Heat Shocks}",
    r"\label{tab:summary_extreme_summer}",
    r"\begin{tabular}{" + header_cols + r"}",
    r"\toprule",
    r" & " + " & ".join(nice_labels[col] for col in cols_display_order) + r" \\",
    r"\midrule"
]

# --- Table body
for _, row in summary_df.iterrows():
    row_vals = []

    # First column: group name as text
    row_vals.append(str(row["Group"]))

    # Remaining columns: numeric formatting
    for col in cols_display_order[1:]:
        val = row[col]

        if isinstance(val, (int, float, np.floating)):
            if col == "% of Total Days":
                # add percent sign, escaped for LaTeX
                cell = f"{val:.2f}\\%"
            else:
                cell = f"{val:.2f}"
        else:
            cell = str(val)

        row_vals.append(cell)

    lines.append(" & ".join(row_vals) + r" \\")

lines += [
    r"\bottomrule",
    r"\end{tabular}",
    r"\begin{minipage}{0.85\linewidth}",
    r"\tiny",
    r"\setstretch{0.9}",
    r"\textit{Notes:} This table presents summary statistics for extreme summer heat shocks",
    r" for all days, winter days (December-Febuary), spring days (March-May), summer days (June-August), and fall (September-November). Extreme days are defined as those where the",
    r" average temperature is more than 3 standard deviations above the long-run norm.",
    r" Panel reports the mean, standard deviation, minimum, number of observations, and the",
    r" share of days classified as extreme within each group.",
    r"\end{minipage}",
    r"\end{table}"
]

with open(out_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print(f"LaTeX table saved to:\n{out_path}")


             Group   Mean    SD    Min  N Obs  % of Total Days
0         All Days  25.40  5.71   7.51   1024             0.12
1  All Winter Days  16.99  3.51   7.51     99             0.05
2  All Spring Days  21.33  3.41   9.89    138             0.06
3  All Summer Days  29.50  2.96  21.39    410             0.18
4    All Fall Days  24.65  5.30  15.45    377             0.20
LaTeX table saved to:
C:\Users\20234503\Desktop\Research\Weather and Strikes\Project\Results_2\Appendix\summary_extreme_all.tex
